In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Benchmark models
models = {
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000),
    'NaiveBayes': MultinomialNB(),
    'SVC': SVC(random_state=42, probability=True)
}

def fit_model(model, train_path):
    """Fit pipeline on train CSV (path provided)."""
    df_train = pd.read_csv(train_path)
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))), 
        ('clf', model)
    ])
    pipe.fit(df_train['text'], df_train['label'])
    return pipe

def score_model(model_pipe, data_path):
    """Score on given data path."""
    df = pd.read_csv(data_path)
    preds = model_pipe.predict(df['text'])
    probs = model_pipe.predict_proba(df['text'])[:, 1]
    return {
        'accuracy': accuracy_score(df['label'], preds),
        'f1': f1_score(df['label'], preds), 
        'probs': probs
    }

def evaluate_predictions(y_true, preds, probs=None):
    """Classification report."""
    print(classification_report(y_true, preds))
    if probs is not None:
        print(f"Accuracy: {accuracy_score(y_true, preds):.3f}")

def validate_model(model_name, train_path='train.csv', val_path='validation.csv'):
    """Validate on custom train/val paths."""
    model = models[model_name]
    pipe = fit_model(model, train_path)
    
    # Train scores
    train_scores = score_model(pipe, train_path)
    print(f"{model_name} Train: Acc {train_scores['accuracy']:.3f}, F1 {train_scores['f1']:.3f}")
    train_df = pd.read_csv(train_path)
    evaluate_predictions(train_df['label'], pipe.predict(train_df['text']))
    
    # Val scores
    val_scores = score_model(pipe, val_path)
    print(f"{model_name} Val: Acc {val_scores['accuracy']:.3f}, F1 {val_scores['f1']:.3f}")
    val_df = pd.read_csv(val_path)
    evaluate_predictions(val_df['label'], pipe.predict(val_df['text']))
    
    return pipe, train_scores, val_scores

def fine_tune_hyperparams(model_name, train_path='train.csv', val_path='validation.csv'):
    """GridSearchCV on train+val paths."""
    from sklearn.model_selection import GridSearchCV
    df_train = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    X_trainval = pd.concat([df_train['text'], df_val['text']])
    y_trainval = pd.concat([df_train['label'], df_val['label']])
    
    base_pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2))), ('clf', models[model_name])])
    
    if model_name == 'LogisticRegression':
        param_grid = {'clf__C': [0.1, 1, 10], 'tfidf__max_features': [3000, 5000]}
    elif model_name == 'SVC':
        param_grid = {'clf__C': [0.1, 1, 10], 'tfidf__max_features': [3000, 5000]}
    else:
        print(f"No tuning for {model_name}")
        return None
    
    grid = GridSearchCV(base_pipe, param_grid, cv=3, scoring='f1', n_jobs=-1)
    grid.fit(X_trainval, y_trainval)
    print(f"Best params {model_name}: {grid.best_params_} (CV F1: {grid.best_score_:.3f})")
    return grid.best_estimator_

def benchmark_test(train_path='train.csv', test_path='test.csv'):
    """Benchmark on custom paths, pick best F1."""
    best_f1, best_model, best_pipe = 0, None, None
    test_df = pd.read_csv(test_path)
    
    for name in models:
        pipe = fit_model(models[name], train_path)
        scores = score_model(pipe, test_path)
        print(f"{name} Test: Acc {scores['accuracy']:.3f}, F1 {scores['f1']:.3f}")
        evaluate_predictions(test_df['label'], pipe.predict(test_df['text']), scores['probs'])
        
        if scores['f1'] > best_f1:
            best_f1, best_model, best_pipe = scores['f1'], name, pipe
    
    print(f"\nBest: {best_model} (Test F1: {best_f1:.3f})")
    return best_pipe


In [ ]:
# Default paths
validate_model('LogisticRegression')
benchmark_test()

# Custom paths (your case)
train_p = r"C:\Users\Ahana\Documents\AML_Assignment1\train.csv"
val_p = r"C:\Users\Ahana\Documents\AML_Assignment1\validation.csv"
test_p = r"C:\Users\Ahana\Documents\AML_Assignment1\test.csv"

validate_model('SVC', train_p, val_p)
tuned_pipe = fine_tune_hyperparams('LogisticRegression', train_p, val_p)
best_pipe = benchmark_test(train_p, test_p)


LogisticRegression Train: Acc 0.977, F1 0.907
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      2894
           1       1.00      0.83      0.91       448

    accuracy                           0.98      3342
   macro avg       0.99      0.92      0.95      3342
weighted avg       0.98      0.98      0.98      3342

LogisticRegression Val: Acc 0.966, F1 0.855
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       965
           1       1.00      0.75      0.85       150

    accuracy                           0.97      1115
   macro avg       0.98      0.87      0.92      1115
weighted avg       0.97      0.97      0.96      1115

LogisticRegression Test: Acc 0.969, F1 0.867
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       966
           1       1.00      0.77      0.87       149

    accuracy                           0.97    